<a href="https://colab.research.google.com/github/aarif-123/House-Price-prediciton/blob/main/HousePrice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# ================================
# 1. Imports
# ================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from xgboost import XGBRegressor

# ================================
# 2. Load Data
# ================================
df = pd.read_csv('train.csv')

# ================================
# 3. Basic Cleaning
# ================================
df = df.drop(columns=['Id'])  # Not useful

# ================================
# 4. Feature Engineering
# ================================
df['HouseAge'] = 2024 - df['YearBuilt']
df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath'])

# ================================
# 5. Target Transform
# ================================
df['SalePrice_log'] = np.log1p(df['SalePrice'])

# ================================
# 6. Split
# ================================
X = df.drop(['SalePrice', 'SalePrice_log'], axis=1)
y = df['SalePrice_log']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ================================
# 7. Column Types
# ================================
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# ================================
# 8. Preprocessing Pipelines
# ================================
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# ================================
# 9. Model Pipeline
# ================================
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])

# ================================
# 10. Train
# ================================
model.fit(X_train, y_train)

# ================================
# 11. Predict
# ================================
y_pred_log = model.predict(X_test)

y_pred = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)

# ================================
# 12. Evaluation
# ================================
r2 = r2_score(y_test_original, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred))

print("R²:", round(r2, 4))
print("RMSE:", round(rmse, 2))

# Accuracy within 10%
percentage_error = np.abs((y_test_original - y_pred) / y_test_original)
accuracy_10 = np.mean(percentage_error < 0.10)

print("Accuracy (within 10%):", round(accuracy_10 * 100, 2), "%")

R²: 0.9066
RMSE: 26772.97
Accuracy (within 10%): 70.21 %
